In [1]:
from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient, query
from pathlib import Path
import os
import pandas as pd
from src.schemas import ProposerResponse, AgentResponse, ToolGeneratorResponse

dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
csv_path = os.path.join(dataset_path, "officeqa.csv")
file_paths = Path("../transformed/")
system_prompt = {"type": "preset", "preset": "claude_code", "append": "You are an expert analyst working at a top office firm. You are given a question and a set of documents. Your job is to answer the question based on the documents. You are also given a set of tools that you can use to answer the question. You can use the tools to read the documents, write the answer, or run code."}
available_tools = ["Read", "Write", "Bash", "Glob", "Grep", "Edit", "WebFetch", "WebSearch", "TodoWrite", "BashOutput"]

cwd = os.getcwd()

train_data = pd.read_csv(csv_path)
hard_data = train_data[train_data['difficulty'] == 'hard']
easy_data = train_data[train_data['difficulty'] == 'easy']


In [27]:
sample_question = hard_data[hard_data.uid == "UID0111"] #.sample(10, random_state=55)
question = sample_question.question.values[0]
answer = sample_question.answer.values[0]
difficulty = sample_question.difficulty.values[0]

In [29]:
print(answer)

[-1832816, -2049753, 216937]


In [30]:
agent_output_format = {
    "type": "json_schema",
    "schema": AgentResponse.model_json_schema()
}
options = ClaudeAgentOptions(
    system_prompt=system_prompt,
    output_format=agent_output_format,
    allowed_tools=available_tools,
    permission_mode='acceptEdits',
    add_dirs=[file_paths],
    cwd=cwd
)

In [31]:
async with ClaudeSDKClient(options) as client:                                                                                             
    await client.query(question)                                                                                                           
    output = [msg async for msg in client.receive_response()] 

In [32]:
print(output[-1].structured_output['reasoning'])

I obtained U.S. federal treasury nominal total receipts and total outlays data for fiscal years 2010-2024 from The American Presidency Project, which sources its data from official OMB historical tables.

The data (in billions of dollars):
- FY 2010: Receipts $2,162.7B, Outlays $3,457.1B
- FY 2011: Receipts $2,303.5B, Outlays $3,603.1B
- FY 2012: Receipts $2,450.0B, Outlays $3,526.6B
- FY 2013: Receipts $2,775.1B, Outlays $3,454.9B
- FY 2014: Receipts $3,021.5B, Outlays $3,506.3B
- FY 2015: Receipts $3,249.9B, Outlays $3,691.9B
- FY 2016: Receipts $3,268.0B, Outlays $3,852.6B
- FY 2017: Receipts $3,316.2B, Outlays $3,981.6B
- FY 2018: Receipts $3,329.9B, Outlays $4,109.0B
- FY 2019: Receipts $3,463.4B, Outlays $4,447.0B
- FY 2020: Receipts $3,421.2B, Outlays $6,550.4B
- FY 2021: Receipts $3,580.8B, Outlays $7,249.5B
- FY 2022: Receipts $4,174.2B, Outlays $6,011.1B
- FY 2023: Receipts $4,641.0B, Outlays $6,013.0B
- FY 2024: Receipts $4,827.8B, Outlays $6,186.8B

I converted these to mil

In [35]:
print(output[-1].structured_output['final_answer'])

[-1359000, -2013649, 654649]


In [36]:
print(answer)

[-1832816, -2049753, 216937]


In [37]:
proposer_system_prompt = {
    "type": "preset",
    "preset": "claude_code",
    "append": "You are an expert agent analyst that is tasked with proposing tools (in the form of tool calls or python code) or skills (in the form of agent skills) that you believe, if added, will help the agent answer the question better. You will be presented with the agent's trace and answer to the question aswell as the ground truth answer. You should also provide a justification for your choice of tool or skill."
}
proposer_output_format = {
            "type": "json_schema",
            "schema": ProposerResponse.model_json_schema()
        }
proposer_options = ClaudeAgentOptions(
    output_format=proposer_output_format,
    system_prompt=proposer_system_prompt,
    # allowed_tools=available_tools,
)

proposer_query = f"Agent trace: {str(output)}\n\nAgent Answer: {output[-1].result}\n\nGround Truth: {answer}"

In [38]:
async with ClaudeSDKClient(options=proposer_options) as client:                                                                                             
    await client.query(proposer_query)                                                                                                           
    proposer_output = [msg async for msg in client.receive_response()] 

In [39]:
from pprint import pprint
pprint(proposer_output[-1].structured_output['justification'])

("The agent's answer differs from the ground truth primarily due to data "
 'accuracy issues. The agent used data from The American Presidency Project '
 'which appears to be less precise than the official Treasury data. When the '
 'agent tried to access the official fiscaldata.treasury.gov API, it received '
 "400 errors because: (1) It didn't know the correct table endpoint (tried "
 'mts_table_1 and mts_table_4 but may have needed mts_table_5 or another '
 'endpoint for the summary data), (2) The filter query syntax was incorrect, '
 '(3) The PDF parsing of official Treasury documents failed. A dedicated '
 'Treasury Fiscal Data API tool would provide the exact official figures '
 "needed for accurate HP filter analysis. The discrepancy between the agent's "
 'actual balance (-1,359,000 million) and ground truth (-1,832,816 million) '
 'represents a ~$474 billion difference, indicating the underlying data was '
 'incorrect. Having a tool that reliably fetches official Treasury data

In [40]:
pprint(proposer_output[-1].structured_output['proposed_tool_or_skill'])

('Treasury Fiscal Data API Tool - A specialized tool that can directly query '
 "the U.S. Treasury's Fiscal Data API (api.fiscaldata.treasury.gov) to "
 'retrieve official financial data. This tool should: (1) Know the correct API '
 'endpoints for the Monthly Treasury Statement dataset (e.g., '
 '/v1/accounting/mts/mts_table_5 for summary of receipts, outlays, and '
 'deficit/surplus), (2) Handle the proper query parameter format for filtering '
 'by fiscal year and selecting September (end of fiscal year) data, (3) Parse '
 'the response to extract total receipts and total outlays in millions of '
 'dollars with full precision, (4) Return structured data suitable for '
 'numerical analysis. The agent encountered 400 errors when trying to query '
 "the API directly because it didn't know the correct table endpoints and "
 'filter syntax.')


In [42]:
tool_generator_system_prompt = {
    "type": "preset",
    "preset": "claude_code",
    "append": "You are an expert skill generator that is tasked with writing python code to add a tool to a ClaudeCode agent through the SDK. You will be given a proposed tool or skill and your task is to implement that skill following the skill-creator skill that's available."
}
tool_generator_output_format = {
            "type": "json_schema",
            "schema": ToolGeneratorResponse.model_json_schema()
        }
tool_generator_options = ClaudeAgentOptions(
    output_format=tool_generator_output_format,
    system_prompt=tool_generator_system_prompt,
    setting_sources=["user", "project"],  # Load Skills from filesystem
    allowed_tools=available_tools + ["Skill"],
    permission_mode='acceptEdits',
    cwd=cwd
)

tool_generator_query = f"Proposed tool or skill: {proposer_output[-1].structured_output['proposed_tool_or_skill']}\n\nJustification: {proposer_output[-1].structured_output['justification']}"

In [43]:
async with ClaudeSDKClient(options=tool_generator_options) as client:                                                                                             
    await client.query(tool_generator_query)                                                                                                           
    tool_generator_output = [msg async for msg in client.receive_response()] 

In [47]:
print(tool_generator_output[-1].structured_output['reasoning'])

I created a comprehensive Treasury Fiscal Data API skill to address the original problem where the agent received 400 errors when trying to query the Treasury API. The key issues were:

1. **Incorrect endpoint paths**: The agent didn't know which MTS table to use (mts_table_9 is best for summary totals)
2. **Wrong field names**: The API uses specific field names like `current_fytd_rcpt_outly_amt` not generic names
3. **Filter syntax errors**: The API requires specific syntax like `filter=record_fiscal_year:eq:2023,record_calendar_month:eq:09`
4. **Data parsing complexity**: Table 9 uses parent-child relationships to distinguish receipts vs outlays totals

The skill I created:
- Provides a Python script (`treasury_api.py`) that correctly queries the Treasury API with proper endpoints and filters
- Uses the `--dataset mts_summary` option for easy access to fiscal year totals (receipts, outlays, deficit/surplus)
- Handles the complex parent-child relationship in Table 9 to correctly ident

In [49]:
agent_output_format = {
    "type": "json_schema",
    "schema": AgentResponse.model_json_schema()
}
next_iteration_options = ClaudeAgentOptions(
    system_prompt=system_prompt,
    output_format=agent_output_format,
    allowed_tools=available_tools + ["Skill"],
    setting_sources=["user", "project"],  # Load Skills from filesystem
    permission_mode='acceptEdits',
    add_dirs=[file_paths],
    cwd=cwd
)

In [50]:
async with ClaudeSDKClient(options=next_iteration_options) as client:                                                                                             
    await client.query(question)                                                                                                           
    next_iteration_output = [msg async for msg in client.receive_response()] 

In [52]:
print(next_iteration_output[-1].structured_output['reasoning'])


I used U.S. federal treasury nominal data from the OMB Historical Tables (Budget of the U.S. Government) for fiscal years 2010-2024. The data includes:

- Total Receipts and Total Outlays for each fiscal year in millions of dollars

I applied the Hodrick-Prescott (HP) filter with smoothing parameter λ=100 separately to:
1. The receipts series → obtaining trend receipts for each year
2. The outlays series → obtaining trend outlays for each year

For each fiscal year, I computed:
- Actual Balance = Receipts - Outlays
- Structural Balance = Trend Receipts - Trend Outlays

For FY 2024:
- Actual Receipts: $4,919,883 million
- Actual Outlays: $6,750,856 million
- Trend Receipts (HP filtered): $4,776,993 million
- Trend Outlays (HP filtered): $6,826,447 million

FY 2024 Results:
1. Actual Balance = 4,919,883 - 6,750,856 = -1,830,973 million
2. Structural Balance = 4,776,993 - 6,826,447 = -2,049,454 million
3. Absolute Gap = |-1,830,973 - (-2,049,454)| = 218,481 million

All values rounded to 

In [53]:
print(next_iteration_output[-1].structured_output['final_answer'])

[-1830973, -2049454, 218481]


In [54]:
print(answer)

[-1832816, -2049753, 216937]


In [55]:
print(output[-1].structured_output['final_answer'])

[-1359000, -2013649, 654649]
